In [ ]:
import numpy as np

import pandas as pd

import csv

x1 = []
x2 = []
pairs = []
y = []



# Data format: CSV with header: pair_id,w1,w2,LED
# Indices inside w1 and w2 are separated by semicolon ';'

def readdata():

   with open('data/processed/db3_processed.csv') as fp:

      reader = csv.reader(fp,delimiter=',')

      next(reader) # Skip header row

      for row in reader:

        # Skip rows containing out-of-vocabulary characters (?)
        if '?' in row[1] or '?' in row[2]:
            continue

        temp1 = row[1].split(';')

        temp1 = [int(x) for x in temp1]

        x1.append(temp1)


        temp2 = row[2].split(';')
        temp2 = [int(x) for x in temp2]

        x2.append(temp2)

        pairs.append([temp1, temp2])


        y.append(int(row[3]))


   fp.close()


   return (np.array(x1,dtype=object),np.array(x2,dtype=object),np.array(pairs,dtype=object),np.array(y))


(x1,x2,pairs,y) = readdata()



In [ ]:
# Data split into train, validation and test

from sklearn.model_selection import train_test_split
import tensorflow as tf

trainX1,testX1,trainX2,testX2,trainy,testy = train_test_split(x1,x2,y,test_size=0.30, random_state=42)
trainX1,valX1,trainX2,valX2,trainy,valy = train_test_split(trainX1,trainX2,trainy,test_size=0.3,random_state=42)

#print(trainX1)
#print(trainX1[11],trainX2[11],trainy[11])
trainX1= tf.ragged.constant(trainX1)
trainX2= tf.ragged.constant(trainX2)
valX1 = tf.ragged.constant(valX1)
valX2 = tf.ragged.constant(valX2)
testX1 = tf.ragged.constant(testX1)
testX2 = tf.ragged.constant(testX2)



In [ ]:
# Architecture

import numpy as np

from sklearn.model_selection import train_test_split

import tensorflow as tf

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Conv1D, Conv2D, GlobalMaxPool1D, Dense, Flatten,Dropout
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras import layers, Model


# Define the custom function for the Lambda layer to be used by create_model and load_model
def abs_difference_for_lambda(tensors):
    return tf.math.abs(tensors[0] - tensors[1])


# Make embeddings for each UNICODE

embedding_dim = 64 # try with 16, 24, 32, 64, 128
#max_seq_length = 20 # pad words with zeros for words of lesser length

# Define the shared model
def create_model():

    x = Sequential()
    x.add(Embedding(60, embedding_dim,input_shape=(None,),mask_zero=True))
    #x.add(LSTM(400,return_sequences=True))
    # CNN
    x.add(Conv1D(250, kernel_size=3, activation='relu',padding='valid'))
    #x.add(Conv1D(400, kernel_size=5, activation='relu'))

    x.add(GlobalMaxPool1D()) # GlobalMinPool1D() or GlobalAvgPool1D()
    x.add(Flatten())
    x.add(Dense(250, activation='relu'))
    #x.add(Dropout(0.3))
    #x.add(Dense(1, activation='sigmoid'))

    # LSTM
    #x.add(LSTM(400))

    shared_model = x

    left_input = Input(shape=(None,))#,dtype='float32')
    right_input = Input(shape=(None,))#,dtype='float32')

    left_output = shared_model(left_input)
    right_output = shared_model(right_input)

    # Use the named function directly instead of an anonymous lambda
    combine_vec1 = layers.Lambda(abs_difference_for_lambda, output_shape=(250,))([left_output,right_output])
    #combine_vec2 = tf.keras.layers.Concatenate(axis=-1)([left_output,right_output])

    output = layers.Dense(1, activation='linear')(combine_vec1)

    model = Model(inputs=[left_input, right_input], outputs=output)
    shared_model.summary()
    model.summary()

    return model

#model.compile(loss='mean_squared_error', optimizer=tf.keras.optimizers.Adam(), metrics=['mse'])

model = create_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 250)       │    684,740 │ input_layer_1[0]… │
│ (Sequential)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 250)       │          0 │ sequential[0][0], │
│                     │                   │            │ sequential[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        251 │ lambda[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 684,991 (2.61 MB)

 Trainable params: 684,991 (2.61 MB)

...

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, None, 250)      │        80,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, None, 400)      │       500,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 400)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 250)            │       100,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 684,740 (2.61 MB)

 Trainable params: 684,740 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training

import keras
from keras import backend as K
K.clear_session()

import numpy as np
np.random.seed(1)

import random as rn
rn.seed(1)

import tensorflow as tf
tf.random.set_seed(1)

from keras.optimizers import Adam
from keras.models import load_model
from keras.utils import plot_model
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.optimizers import SGD
import pandas as pd


from numpy import array
from numpy import argmax
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

import argparse
import locale
import os
import sys
import matplotlib.pyplot as plt

os.environ['PYTHONHASHSEED'] = '0'

# create our Convolutional Neural Network and then compile the model


#model = create_model()

#model=load_model('bestweights.h5')
opt=Adam(learning_rate=0.001,beta_1=0.99,beta_2=0.999,epsilon=1e-06,amsgrad=True)
model.compile(loss='mean_squared_error', optimizer=tf.keras.optimizers.Adam(), metrics=['mse'])


# train the model
print("[INFO] training model...")
es = EarlyStopping(monitor = 'val_loss', mode = 'min', verbose=1,patience=10)
cp = ModelCheckpoint(filepath="bestweights.h5",monitor = 'val_loss', mode='min',verbose = 1, save_best_only = True)
#hist=model.fit(traindata, trainLabelsY,validation_data=(valdata[0], valLabelsY[0]),  epochs=100, batch_size=24,callbacks=[es,cp])
hist=model.fit([trainX1.to_tensor(default_value=0),trainX2.to_tensor(default_value=0)], trainy,validation_data=([valX1.to_tensor(default_value=0),valX2.to_tensor(default_value=0)], valy),  epochs=100, batch_size=12,callbacks=[es,cp])

model.save("finalweights.h5")

# plot the model, LR - horizontal, TP - vertical
plot_model(model, 'siamese_model.png', show_shapes=True, rankdir='LR')
print(model.summary())

df_history = pd.DataFrame(hist.history)
df_history.to_csv("history.csv", index=False, header=True, sep='\t')

#print(hist.history.keys())
plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.legend(['Training','Validation'])
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.savefig("accplot.png",dpi = 300)


In [ ]:
from tensorflow.keras.models import load_model

import tensorflow as tf # Ensure tf is available for the Lambda layer globally

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import matplotlib.pyplot as plt
import os
import cv2
from imutils import build_montages
import random
import pandas as pd
import numpy as np

from numpy import array
from numpy import argmax

# The function abs_difference_for_lambda is now defined in cell 2Hd6ET2Khk3H
# and is accessible globally. Its local definition here is no longer needed.

def test_model(testdata,testLabelsY,weights_file):    # make predictions on the testing data
    # Pass the custom function (and tf) to custom_objects
    # The function abs_difference_for_lambda will be resolved from the global scope after cell 2Hd6ET2Khk3H is executed.
    model=load_model(weights_file, safe_mode=False, custom_objects={'tf': tf, 'abs_difference_for_lambda': abs_difference_for_lambda})

    preds = model.predict(testdata)

    #print(preds)
    print(np.sqrt(np.mean((testLabelsY-preds)**2)))




test_model([trainX1.to_tensor(default_value=0),trainX2.to_tensor(default_value=0)],trainy,'finalweights.h5')
test_model([valX1.to_tensor(default_value=0),valX2.to_tensor(default_value=0)],valy,'finalweights.h5')
test_model([testX1.to_tensor(default_value=0),testX2.to_tensor(default_value=0)],testy,'finalweights.h5')
